In [11]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [12]:
result_path = Path("../results/downstream_task")
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
method_name_replacer = {"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                            # "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "mrs-forest": "MRS", 
                       # "fw-mrs-temperature-svm": "FW-MRS-SVM", 
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-negative": "FW-MRS$_{Neg}$",
                          }
data_set_replacer = {"folktables_employment": "Employment", "folktables_income": "Income",
                               "breast_cancer": "Breast Cancer", "hr_analytics": "HR Analytic", "loan_prediction": "Loan",
                               "diabetes": "Diabetes", "german_credit": "German Credit", "bank_marketing": "Bank Marketing"}

In [13]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [14]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.870601,0.010491,0.826992,0.017307,less_positive_class,0.1,0.000000,0.000000
1,KMM,folktables_employment,0.856910,0.012538,0.809240,0.020048,less_positive_class,0.1,0.000000,0.000000
2,PSA,folktables_employment,0.867229,0.011378,0.823210,0.017361,less_positive_class,0.1,0.020000,0.140000
3,MRS,folktables_employment,0.869439,0.010185,0.825394,0.016376,less_positive_class,0.1,283.100000,114.960384
4,FW-MRS,folktables_employment,0.864510,0.011807,0.819362,0.018359,less_positive_class,0.1,378.400000,89.595982
5,FW-MRS$_{Neg}$,folktables_employment,0.874457,0.009650,0.831544,0.015783,less_positive_class,0.1,296.914894,102.415264
6,Uniform,folktables_income,0.838084,0.012878,0.788634,0.020309,less_positive_class,0.1,0.000000,0.000000
7,KMM,folktables_income,0.819722,0.014641,0.764636,0.020496,less_positive_class,0.1,0.000000,0.000000
8,PSA,folktables_income,0.831327,0.012480,0.781005,0.019604,less_positive_class,0.1,0.020000,0.140000
9,MRS,folktables_income,0.837534,0.012058,0.788396,0.018049,less_positive_class,0.1,267.600000,93.065783


In [15]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auroc_values = []
            std_auroc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
\\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.871\pm0.01$ & $0.857\pm0.01$ & $0.867\pm0.01$ & $0.869\pm0.01$ & $0.865\pm0.01$ & $0.874\pm0.01$ \\
Income & $0.838\pm0.01$ & $0.82\pm0.01$ & $0.831\pm0.01$ & $0.838\pm0.01$ & $0.834\pm0.01$ & $0.837\pm0.01$ \\
Breast Cancer & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.987\pm0.01$ & $0.987\pm0.0$ \\
HR Analytic & $0.753\pm0.02$ & $0.749\pm0.02$ & $0.751\pm0.02$ & $0.752\pm0.02$ & $0.751\pm0.02$ & $0.749\pm0.02$ \\
Loan & $0.658\pm0.08$ & $0.607\pm0.09$ & $0.628\pm0.1$ & $0.643\pm0.09$ & $0.624\pm0.09$ & $0.673\pm0.09$ \\
Diabetes & $0.791\pm0.02$ & $0.78\pm0.02$ & $0.788\pm0.02$ & $0.79\pm0.02$ & $0.788\pm0.02$ & $0.786\pm0.03$ \\
German Credit & $0.667\pm0.05$ & $0.644\pm0.05$ & $0.663\pm0.05$ & $0.668\pm0.05$ & $0.656\pm0.06$ & $0.667\pm0.05$ \\
Bank Marketing & $0.847\pm0.02$ & $0.829\pm0.03$ & $0.838\pm0.03$ & $0.844\pm0.02$ & $0.838\pm0.03$ & $0.85\pm0.02$ \\




In [16]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auprc_values = []
            std_auprc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                    mean_auprc_values.append(np.round(mean_auprc, 3))

                    std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                    std_auprc_values.append(np.round(std_auprc, 2))
                except IndexError:
                    mean_auprc_values.append(0)
                    std_auprc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ \
& ${mean_auprc_values[5]}\\pm{std_auprc_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Employment & $0.827\pm0.02$ & $0.809\pm0.02$ & $0.823\pm0.02$ & $0.825\pm0.02$ & $0.819\pm0.02$ & $0.832\pm0.02$ & \\
Income & $0.789\pm0.02$ & $0.765\pm0.02$ & $0.781\pm0.02$ & $0.788\pm0.02$ & $0.785\pm0.02$ & $0.786\pm0.02$ & \\
Breast Cancer & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.995\pm0.0$ & $0.994\pm0.0$ & $0.994\pm0.0$ & \\
HR Analytic & $0.455\pm0.03$ & $0.448\pm0.03$ & $0.455\pm0.03$ & $0.453\pm0.03$ & $0.452\pm0.03$ & $0.449\pm0.04$ & \\
Loan & $0.795\pm0.05$ & $0.775\pm0.06$ & $0.78\pm0.06$ & $0.792\pm0.05$ & $0.781\pm0.06$ & $0.801\pm0.06$ & \\
Diabetes & $0.371\pm0.04$ & $0.356\pm0.04$ & $0.366\pm0.04$ & $0.368\pm0.04$ & $0.365\pm0.04$ & $0.365\pm0.04$ & \\
German Credit & $0.461\pm0.06$ & $0.437\pm0.06$ & $0.458\pm0.06$ & $0.462\pm0.06$ & $0.447\pm0.07$ & $0.456\pm0.06$ & \\
Bank Marketing & $0.466\pm0.05$ & $0.441\pm0.06$ & $0.453\pm0.05$ & $0.468\pm0.05$ & $0.455\pm0.05$ & $0.469\pm0.05$ & \\




In [17]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.871,0.010,0.827,0.017,less_positive_class,0.1,0.000,0.000
1,KMM,folktables_employment,0.857,0.013,0.809,0.020,less_positive_class,0.1,0.000,0.000
2,PSA,folktables_employment,0.867,0.011,0.823,0.017,less_positive_class,0.1,0.020,0.140
3,MRS,folktables_employment,0.869,0.010,0.825,0.016,less_positive_class,0.1,283.100,114.960
4,FW-MRS,folktables_employment,0.865,0.012,0.819,0.018,less_positive_class,0.1,378.400,89.596
5,FW-MRS$_{Neg}$,folktables_employment,0.874,0.010,0.832,0.016,less_positive_class,0.1,296.915,102.415
6,Uniform,folktables_income,0.838,0.013,0.789,0.020,less_positive_class,0.1,0.000,0.000
7,KMM,folktables_income,0.820,0.015,0.765,0.020,less_positive_class,0.1,0.000,0.000
8,PSA,folktables_income,0.831,0.012,0.781,0.020,less_positive_class,0.1,0.020,0.140
9,MRS,folktables_income,0.838,0.012,0.788,0.018,less_positive_class,0.1,267.600,93.066


In [18]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.round(3).groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS,4.5000,4.3750
FW-MRS$_{Neg}$,3.0625,3.0000
KMM,5.3750,5.4375
MRS,2.1250,2.1875
PSA,4.0000,3.8750
Uniform,1.9375,2.1250
